In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score

# ==============================================================================
# PART 1: CONFIGURATION & EXPLANATION
# ==============================================================================
# Hyperparameters
BATCH_SIZE = 64
LEARNING_RATE = 0.1  # Hyperparameter
EPOCHS = 10          # Hyperparameter

# Device configuration (Use GPU if available, otherwise CPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

Running on device: cuda:0


In [2]:
# ==============================================================================
# PART 2: DATA PREPARATION (CIFAR-10)
# ==============================================================================
print("Preparing Data...")

# We use standard transformations.
# To help the network learn, we normalize the images (mean 0.5, std 0.5).
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# Load Training Data
trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE,
                                          shuffle=True, num_workers=2)

# Load Test Data
testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE,
                                         shuffle=False, num_workers=2)

# The 10 classes in CIFAR-10
classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

Preparing Data...


100%|██████████| 170M/170M [00:03<00:00, 56.4MB/s]


In [4]:
# ==============================================================================
# PART 3: THE NiN BLOCK
# ==============================================================================
class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()

        # ----------------------------------------------------------------------
        # LAYER 1: The "Spatial" Convolution
        # ----------------------------------------------------------------------
        # This is a standard CNN layer. It looks at a local patch (e.g., 3x3)
        # to find spatial patterns like edges, corners, or textures.
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.bn1 = nn.BatchNorm2d(out_channels) # Normalizes data for faster training

        # ----------------------------------------------------------------------
        # LAYER 2: The First 1x1 Convolution (The "Hidden Layer")
        # ----------------------------------------------------------------------
        # CONCEPT FOR STUDENTS:
        # A 1x1 convolution is mathematically identical to a Dense (Fully Connected)
        # layer applied to every single pixel independently.
        # It takes the 'out_channels' from the previous layer, mixes them using
        # weights, and outputs 'out_channels'. It does NOT look at neighbors.
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # ----------------------------------------------------------------------
        # LAYER 3: The Second 1x1 Convolution (The "Output Layer" of the MLP)
        # ----------------------------------------------------------------------
        # We add another 1x1 layer to make this "pixel-wise MLP" deeper and
        # more non-linear. This increases the network's power without changing
        # the spatial resolution (Height/Width).
        self.conv3 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.bn3 = nn.BatchNorm2d(out_channels)

    def forward(self, x):
        # We use ReLU activation after every convolution to introduce non-linearity.
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.relu(self.bn3(self.conv3(x)))
        return x

In [8]:
# ==============================================================================
# PART 4: THE FULL NETWORK ARCHITECTURE
# ==============================================================================
class NetInNet(nn.Module):
    def __init__(self):
        super(NetInNet, self).__init__()

        # --- BLOCK 1 ---
        # Input: 3 channels (RGB), Output: 96 channels
        # We use a 3x3 spatial kernel.
        self.block1 = NiNBlock(in_channels=3, out_channels=96,
                               kernel_size=3, stride=1, padding=1)

        # MaxPool: Shrinks the grid by half (32x32 -> 16x16)
        self.pool1 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        # Dropout: Randomly zeros neurons to prevent overfitting
        self.drop1 = nn.Dropout(0.5)

        # --- BLOCK 2 ---
        # Input: 96 channels, Output: 256 channels
        # We increase the "depth" (channels) as we go deeper, typical in CNNs.
        self.block2 = NiNBlock(in_channels=96, out_channels=256,
                               kernel_size=3, stride=1, padding=1)

        # MaxPool: Shrinks the grid by half (16x16 -> 8x8)
        self.pool2 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.drop2 = nn.Dropout(0.5)

        # --- BLOCK 3 ---
        # Input: 256 channels, Output: 384 channels
        # We assume the 8x8 grid now contains high-level concepts.
        self.block3 = NiNBlock(in_channels=256, out_channels=384,
                               kernel_size=3, stride=1, padding=1)

        # MaxPool: Shrinks the grid by half (8x8 -> 4x4)
        self.pool3 = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        self.drop3 = nn.Dropout(0.5)

        # --- THE CLASSIFICATION HEAD (Crucial Difference from MLPs) ---
        # Instead of flattening and using a Huge Dense Layer (Parameter Explosion!),
        # we use the "NiN Strategy":

        # 1. Project to Classes:
        # We use a 1x1 Conv to squash 384 channels down to 10 channels.
        # Why 10? Because we have 10 classes (Dog, Cat, etc.).
        # This creates 10 "Heatmaps" of size 4x4.
        self.classifier_conv = nn.Conv2d(384, 10, kernel_size=1)

        # 2. Global Average Pooling (GAP):
        # We will average the 4x4 heatmap for "Dog" into a single score.
        # (Implemented in forward function)

    def forward(self, x):
        # Pass through the feature extraction blocks
        x = self.drop1(self.pool1(self.block1(x)))
        x = self.drop2(self.pool2(self.block2(x)))
        x = self.drop3(self.pool3(self.block3(x)))

        # --- CLASSIFICATION ---

        # Step A: Create the Class Heatmaps
        # Shape Input:  [Batch, 384, 4, 4]
        # Shape Output: [Batch, 10,  4, 4]
        x = self.classifier_conv(x)

        # Step B: Global Average Pooling
        # We average across Height (dim 2) and Width (dim 3).
        # Shape Input:  [Batch, 10, 4, 4]
        # Shape Output: [Batch, 10, 1, 1]
        x = F.avg_pool2d(x, kernel_size=x.size()[2:])

        # Step C: Flatten
        # We just remove the 1x1 dimensions to get a vector of 10 scores.
        # Shape Output: [Batch, 10]
        x = x.view(x.size(0), -1)

        return x

In [6]:
# ==============================================================================
# PART 5: ARCHITECTURE INSPECTION (The "X-Ray" for Students)
# ==============================================================================
def inspect_model_shapes():
    print("\n" + "="*50)
    print("ARCHITECTURAL X-RAY: TRACING SHAPES")
    print("="*50)

    # Create a dummy image: Batch=1, Channels=3, Height=32, Width=32
    dummy_input = torch.randn(1, 3, 32, 32).to(device)
    model = NetInNet().to(device)

    # We will hook into the model to print shapes as data passes through
    print(f"1. Input Image:           {list(dummy_input.shape)}")

    x = model.block1(dummy_input)
    print(f"2. After Block 1 (Conv):  {list(x.shape)} -> (Notice channels grew to 96)")
    x = model.pool1(x)
    print(f"3. After Pool 1:          {list(x.shape)} -> (Size shrunk to 16x16)")

    x = model.block2(x)
    x = model.pool2(x)
    print(f"4. After Block 2 + Pool:  {list(x.shape)} -> (Size 8x8, Channels 256)")

    x = model.block3(x)
    x = model.pool3(x)
    print(f"5. After Block 3 + Pool:  {list(x.shape)} -> (Size 4x4, Channels 384)")

    # The Critical Head Part
    x = model.classifier_conv(x)
    print(f"6. After 1x1 Classifier:  {list(x.shape)} -> (Now we have 10 class maps!)")

    x = F.avg_pool2d(x, kernel_size=x.size()[2:])
    print(f"7. After Global Avg Pool: {list(x.shape)} -> (Spatial dims averaged to 1x1)")

    x = x.view(x.size(0), -1)
    print(f"8. Final Output Vector:   {list(x.shape)} -> (Ready for Softmax)")
    print("="*50 + "\n")

# Run the inspection to show students the dimensions
inspect_model_shapes()


ARCHITECTURAL X-RAY: TRACING SHAPES
1. Input Image:           [1, 3, 32, 32]
2. After Block 1 (Conv):  [1, 96, 32, 32] -> (Notice channels grew to 96)
3. After Pool 1:          [1, 96, 16, 16] -> (Size shrunk to 16x16)
4. After Block 2 + Pool:  [1, 256, 8, 8] -> (Size 8x8, Channels 256)
5. After Block 3 + Pool:  [1, 384, 4, 4] -> (Size 4x4, Channels 384)
6. After 1x1 Classifier:  [1, 10, 4, 4] -> (Now we have 10 class maps!)
7. After Global Avg Pool: [1, 10, 1, 1] -> (Spatial dims averaged to 1x1)
8. Final Output Vector:   [1, 10] -> (Ready for Softmax)

